# Remove/Move/Rename files from S3
- This notebook is to be used to remove files from s3 and can be used to mass remove files from a folder
- This notebook is to be used to move files in s3 and can be used to move/rename files from a folder

In [ ]:
import os

import boto3
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from multiprocessing.pool import Pool

load_dotenv()

## Connect do S3

In [ ]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY')
)

BUCKET = '<the bucket>'
KEY = '<the common key for files>'

# check if bucket exists
list_of_buckets = [r['Name'] for r in s3_client.list_buckets()['Buckets']]
if not BUCKET in list_of_buckets: raise Exception()

## check files to delete or move

In [ ]:
paginator = s3_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET, Prefix=KEY+'ixcard')

objects_to_delete = []
for page in tqdm(pages):
    for obj in page['Contents']:
        objects_to_delete.append(obj['Key'])

counter_uploaded = pd.DataFrame(objects_to_delete)
print(counter_uploaded.shape)
counter_uploaded.head()

## Delete files

### single file
delete a single file frm S3

### multi files using pagination
- when there are more than 1000 files to be deleted, using boto will require pagination because boto only lists 1000 files at the time
- boto3 delete function will also delete 1000 files each time

In [ ]:
paginator = s3_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET, Prefix=KEY)

objects_to_delete = {'Objects': []}

for obj in tqdm(pages.search('Contents')):
    objects_to_delete['Objects'].append({'Key': obj['Key']})

    # flush once aws limit reached
    if len(objects_to_delete['Objects']) >= 1000:
        s3_client.delete_objects(Bucket=BUCKET, Delete=objects_to_delete)
        objects_to_delete = {'Objects': []}
        
if len(objects_to_delete['Objects']) < 1000:
    s3_client.delete_objects(Bucket=BUCKET, Delete=objects_to_delete)

## Move Files

In [ ]:
def copy_file_in_s3(old_key):

    new_key = old_key.replace('<old key>', '<new key>')
    copy_source = {'Bucket': BUCKET, 'Key': old_key}

    s3_client.copy(copy_source, BUCKET, new_key)

### Using paralelism to speed up process

In [ ]:
with Pool() as p:
    with tqdm(total=len(files_to_copy), desc='copy files', ncols=100) as pbar:
        for _ in p.imap_unordered(copy_file_in_s3, files_to_copy):
            pbar.update()

In [ ]:
for f in tqdm(files_to_copy, desc='check copy', ncols=100):
    try:
        s3_client.head_object(Bucket=BUCKET, Key=f.replace('<old key>', '<new key>'))
    except s3_client.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            print('Key does not exist !!!')
        else:
            raise